# 실습 과제 답안: 지역별 식량작물 생산량 분석 대시보드

같은 폴더에 **제공된 CSV 파일**을 이용해 Streamlit 대시보드를 구현합니다.

| 항목 | 내용 |
|------|------|
| 목표 | 지역·작물·연도 선택 → 핵심 통계 / Line·Bar Chart / 원본 표 표시 |
| 데이터 | `식량작물_생산량_정곡.csv` (또는 `식량작물_생산량_정곡__20260808172613.csv`) |
| 기간 | 2016~2025년 (2026년 제외) |
| 실행 | 노트북으로 데이터·로직 확인 후 `streamlit run app.py` |
| 선택 과제 | 10년 증감률, 식량작물별 비교, 지역별 생산량 순위 |


## 0. 라이브러리 설치 (필요 시)

아래 셀은 한 번만 실행하면 됩니다.


In [1]:
# 사용법: 과제에 필요한 패키지를 설치합니다.
%pip install pandas plotly streamlit -q


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. 제공 CSV 파일 구조 확인

통계청형 식량작물 생산량 CSV는 **다중 헤더** 구조입니다.

| 행 | 내용 |
|----|------|
| 0행 | 연도 (`2016`, `2017`, …, `2025`) |
| 1행 | 지표명 (`미곡:생산량 (톤)`, `맥류:면적 (ha)` 등) |
| 2행~ | 지역별 값 (`시도별(1)`, `시도별(2)` + 수치) |

대시보드에서는 **작물별 생산량(톤)** 열만 사용하고, 분석 기간은 **2016~2025**로 제한합니다.


In [2]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path.cwd()

CSV_CANDIDATES = [
    "식량작물_생산량_정곡.csv",
    "식량작물_생산량_정곡__20260808172613.csv",
]

csv_path = next((DATA_DIR / name for name in CSV_CANDIDATES if (DATA_DIR / name).exists()), None)
if csv_path is None:
    raise FileNotFoundError(
        "CSV 파일이 없습니다. "
        "식량작물_생산량_정곡.csv 또는 "
        "식량작물_생산량_정곡__20260808172613.csv 를 같은 폴더에 두세요."
    )

# 사용법: header 없이 읽어 원본 구조를 확인합니다.
raw = pd.read_csv(csv_path, encoding="cp949", header=None)
print("파일:", csv_path.name)
print("원본 shape:", raw.shape)
print("0행(연도) 샘플:", raw.iloc[0, 2:8].tolist())
print("1행(지표) 샘플:", raw.iloc[1, 2:8].tolist())
raw.iloc[:6, :6]


파일: 식량작물_생산량_정곡__20260808172613.csv
원본 shape: (20, 122)
0행(연도) 샘플: ['2016', '2016', '2016', '2016', '2016', '2016']
1행(지표) 샘플: ['합계:면적 (ha)', '합계:생산량 (톤)', '미곡:면적 (ha)', '미곡:생산량 (톤)', '맥류:면적 (ha)', '맥류:생산량 (톤)']


,0,1,2,3,4,5
0,시도별(1),시도별(2),2016,2016,2016,2016
1,시도별(1),시도별(2),합계:면적 (ha),합계:생산량 (톤),미곡:면적 (ha),미곡:생산량 (톤)
2,전국,소계,961792,4706554,778734,4196691
3,서울특별시,소계,138,699,123,630
4,전남광주통합특별시,광주광역시,7057,33322,5191,26431
5,전남광주통합특별시,전라남도,208699,953512,166444,846236


## 2. CSV 전처리 + 장형(long) 데이터 만들기

- `시도별(2)`가 `소계`가 아니면 하위 지역명 사용 (예: 전라남도, 광주광역시)
- `미곡`, `맥류`, `잡곡`, `두류`, `서류`의 **생산량**만 추출
- `-` / 빈 값은 결측으로 처리
- 2016~2025년만 남김


In [3]:
CROPS = ["미곡", "맥류", "잡곡", "두류", "서류"]
YEARS = list(range(2016, 2026))  # 2016~2025


def to_number(value):
    """'-', 빈 값, 쉼표 포함 숫자를 float로 변환합니다."""
    if pd.isna(value):
        return None
    text = str(value).strip().replace(",", "")
    if text in {"", "-", "nan", "None"}:
        return None
    return float(text)


def load_production_data(path: Path) -> pd.DataFrame:
    """통계청형 다중 헤더 CSV를 장형 데이터로 변환합니다."""
    raw = pd.read_csv(path, encoding="cp949", header=None)
    years = raw.iloc[0, 2:].astype(str).str.strip().tolist()
    metrics = raw.iloc[1, 2:].astype(str).str.strip().tolist()
    body = raw.iloc[2:].reset_index(drop=True)

    records = []
    for i in range(len(body)):
        sido1 = str(body.iloc[i, 0]).strip()
        sido2 = str(body.iloc[i, 1]).strip()
        region = sido2 if sido2 != "소계" else sido1

        for j, (year_text, metric) in enumerate(zip(years, metrics)):
            year = int(float(year_text))
            if year not in YEARS:
                continue

            crop = None
            for name in CROPS:
                if metric.startswith(f"{name}:") and "생산량" in metric:
                    crop = name
                    break
            if crop is None:
                continue

            records.append(
                {
                    "지역": region,
                    "연도": year,
                    "작물": crop,
                    "생산량": to_number(body.iloc[i, j + 2]),
                }
            )

    return pd.DataFrame(records).sort_values(["지역", "작물", "연도"]).reset_index(drop=True)


df = load_production_data(csv_path)
print("전처리 후 shape:", df.shape)
print("지역 수:", df["지역"].nunique())
print("작물:", df["작물"].unique().tolist())
print("연도:", sorted(df["연도"].unique().tolist()))
df.head(10)


전처리 후 shape: (900, 4)
지역 수: 18
작물: ['두류', '맥류', '미곡', '서류', '잡곡']
연도: [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]


,지역,연도,작물,생산량
0,강원특별자치도,2016,두류,10008.0
1,강원특별자치도,2017,두류,7740.0
2,강원특별자치도,2018,두류,9048.0
3,강원특별자치도,2019,두류,10952.0
4,강원특별자치도,2020,두류,9560.0
5,강원특별자치도,2021,두류,10899.0
6,강원특별자치도,2022,두류,10054.0
7,강원특별자치도,2023,두류,10428.0
8,강원특별자치도,2024,두류,10333.0
9,강원특별자치도,2025,두류,NaN


## 3. 핵심 통계 계산

선택한 지역·작물에 대해 다음을 계산합니다.

- 선택 연도 생산량
- 10년 평균 생산량
- 10년간 최대 생산량
- (선택) 2016→2025 생산량 증감률


In [4]:
# 사용법: 노트북에서 확인할 기본 선택값입니다. Streamlit에서는 selectbox로 바꿉니다.
region = "전라남도"
crop = "미곡"
year = 2025

region_crop = df[(df["지역"] == region) & (df["작물"] == crop)].copy()
region_crop_valid = region_crop.dropna(subset=["생산량"])

selected_row = region_crop.loc[region_crop["연도"] == year, "생산량"]
selected_prod = float(selected_row.iloc[0]) if not selected_row.empty else None
avg_10y = float(region_crop_valid["생산량"].mean()) if not region_crop_valid.empty else None
max_10y = float(region_crop_valid["생산량"].max()) if not region_crop_valid.empty else None

v2016_row = region_crop.loc[region_crop["연도"] == 2016, "생산량"]
v2025_row = region_crop.loc[region_crop["연도"] == 2025, "생산량"]
v2016 = float(v2016_row.iloc[0]) if not v2016_row.empty and pd.notna(v2016_row.iloc[0]) else None
v2025 = float(v2025_row.iloc[0]) if not v2025_row.empty and pd.notna(v2025_row.iloc[0]) else None

if v2016 and v2016 != 0 and v2025 is not None:
    change_rate = (v2025 - v2016) / v2016 * 100
else:
    change_rate = None


def format_ton(value):
    if value is None or pd.isna(value):
        return "-"
    return f"{value:,.0f}톤"


print(f"[{region} · {crop}]")
print(f"{year}년 생산량 :", format_ton(selected_prod))
print("10년 평균     :", format_ton(avg_10y))
print("최대 생산량   :", format_ton(max_10y))
print(
    "10년 증감률   :",
    f"{change_rate:+.1f}%" if change_rate is not None else "-",
)
region_crop


[전라남도 · 미곡]
2025년 생산량 : 686,504톤
10년 평균     : 751,775톤
최대 생산량   : 846,236톤
10년 증감률   : -18.9%


,지역,연도,작물,생산량
670,전라남도,2016,미곡,846236.0
671,전라남도,2017,미곡,827162.0
672,전라남도,2018,미곡,766022.0
673,전라남도,2019,미곡,725094.0
674,전라남도,2020,미곡,687812.0
675,전라남도,2021,미곡,789650.0
676,전라남도,2022,미곡,742913.0
677,전라남도,2023,미곡,736985.0
678,전라남도,2024,미곡,709368.0
679,전라남도,2025,미곡,686504.0


## 4. Plotly로 연도별 생산량 Line Chart

- X축: `연도`
- Y축: `생산량`
- 제목: `{지역} {작물} 연도별 생산량 변화`


In [5]:
import plotly.express as px

fig_line = px.line(
    region_crop_valid,
    x="연도",
    y="생산량",
    markers=True,
    title=f"{region} {crop} 연도별 생산량 변화",
)
fig_line.update_layout(xaxis_title="연도", yaxis_title="생산량(톤)")
fig_line.update_xaxes(dtick=1)
fig_line.show()


## 5. Plotly로 지역별 생산량 Bar Chart

선택한 연도·작물의 지역별 생산량을 비교합니다. (`전국` 제외)


In [6]:
year_crop = df[
    (df["연도"] == year) & (df["작물"] == crop) & (df["지역"] != "전국")
].copy()
year_crop = year_crop.dropna(subset=["생산량"]).sort_values("생산량", ascending=False)

fig_bar = px.bar(
    year_crop,
    x="생산량",
    y="지역",
    orientation="h",
    text_auto=".0f",
    title=f"{year}년 {crop} 생산량",
)
fig_bar.update_layout(
    xaxis_title="생산량(톤)",
    yaxis_title="지역",
    yaxis={"categoryorder": "total ascending"},
)
fig_bar.show()

# 선택 과제 3: 지역별 생산량 순위
rank_df = year_crop.reset_index(drop=True).copy()
rank_df.insert(0, "순위", range(1, len(rank_df) + 1))
rank_df["생산량(톤)"] = rank_df["생산량"].map(lambda v: f"{v:,.0f}")
rank_df[["순위", "지역", "생산량(톤)"]]


,순위,지역,생산량(톤)
0,1,충청남도,"693,819"
1,2,전라남도,"686,504"
2,3,전북특별자치도,"543,137"
3,4,경상북도,"474,157"
4,5,경기도,"369,148"
5,6,경상남도,"305,247"
6,7,충청북도,"170,973"
7,8,강원특별자치도,"142,509"
8,9,인천광역시,"57,724"
9,10,대구광역시,"25,984"


## 6. (선택) 식량작물별 생산량 비교

특정 지역의 미곡·맥류·잡곡·두류·서류 생산량을 Bar Chart로 비교합니다.


In [7]:
crop_cmp = df[(df["지역"] == region) & (df["연도"] == year)].copy()
crop_cmp = crop_cmp.dropna(subset=["생산량"])
crop_cmp["작물"] = pd.Categorical(crop_cmp["작물"], categories=CROPS, ordered=True)
crop_cmp = crop_cmp.sort_values("작물")

fig_crop = px.bar(
    crop_cmp,
    x="작물",
    y="생산량",
    text_auto=".0f",
    title=f"{region} {year}년 식량작물별 생산량",
    color="작물",
)
fig_crop.update_layout(xaxis_title="작물", yaxis_title="생산량(톤)", showlegend=False)
fig_crop.show()
crop_cmp


,지역,연도,작물,생산량
679,전라남도,2025,미곡,686504.0
669,전라남도,2025,맥류,37510.0


## 7. Streamlit 앱 실행

같은 폴더의 `app.py`에 위 로직이 반영되어 있습니다.

터미널에서 아래 명령으로 실행합니다.

```bash
streamlit run app.py
```

중지: `Ctrl + C`


In [8]:
# 사용법: 노트북에서 앱 파일이 있는지 확인합니다.
app_path = DATA_DIR / "app.py"
print("app.py 존재:", app_path.exists())
print("실행 명령: streamlit run app.py")
print("작업 폴더:", DATA_DIR)


app.py 존재: True
실행 명령: streamlit run app.py
작업 폴더: c:\MyCursorLab\03_농업 데이터 시각화 대시보드 만들기\실습과제\02_지역 농업 통계 데이터 분석 대시보드


## 8. 분석 질문 (예시 답)

1. **2016~2025년 동안 선택한 지역의 식량작물 생산량은 어떻게 변화했는가?**  
   → Line Chart에서 전라남도 미곡은 대체로 감소 추세이며, 연도별로 등락이 있습니다.

2. **선택한 연도에 생산량이 가장 많은 지역은 어디인가?**  
   → Bar Chart / 순위 표에서 확인합니다. (예: 2025년 미곡 기준 상위 지역)

3. **미곡, 맥류, 잡곡, 두류, 서류 중 어떤 작물의 생산량이 가장 많은가?**  
   → 대부분의 지역에서 **미곡** 생산량이 가장 큽니다.

> 참고: 2025년은 일부 작물(잡곡·두류·서류)이 `-`(미공표)일 수 있습니다.
